In [1]:
#Chat am I muted?
import moviepy
from moviepy import *
from moviepy.video import *
import copy
'''
# loading video gfg
clip = VideoFileClip("../Data/modelUnderwaters.mp4")


# getting subclip from it
clip = clip.subclipped(0, 4)

# applying speed effect
effect = vfx.AccelDecel(1, abruptness=0, soonness=1)

clip = effect.apply(clip)



# showing final clip
#clip.display_in_notebook()


#from conf import SAMPLE_INPUTS, SAMPLE_OUTPUTS

length = 10

clip1 = VideoFileClip("../Data/modelUnderwaters.mp4").subclipped(0, 0 + 5)
clip2 = VideoFileClip("../Data/flipturnHQ.mp4").subclipped(0, 0 + 5)


combined = clips_array([[clip1], [clip2]])

effect = vfx.Margin(left=960, right=960)
combined = effect.apply(combined)
effect = vfx.Resize((1920,1080) )
combined = effect.apply(combined)

#combined = combined.resize(height=1920)


effect = vfx.Crop(x1=1166.6, y1=0, x2=2246.6, y2=1920)
#clip = effect.apply(clip)
#combined = combined.crop(x1=1166.6, y1=0, x2=2246.6, y2=1920)

#combined = combined.with_effects(vfx.Resize((1920,1080)), vfx.Margin(960))
combined.display_in_notebook
#combined.write_videofile('test.mp4')

combined.write_videofile("../outputs/result.mp4")

'''

'\n# loading video gfg\nclip = VideoFileClip("../Data/modelUnderwaters.mp4")\n\n\n# getting subclip from it\nclip = clip.subclipped(0, 4)\n\n# applying speed effect\neffect = vfx.AccelDecel(1, abruptness=0, soonness=1)\n\nclip = effect.apply(clip)\n\n\n\n# showing final clip\n#clip.display_in_notebook()\n\n\n#from conf import SAMPLE_INPUTS, SAMPLE_OUTPUTS\n\nlength = 10\n\nclip1 = VideoFileClip("../Data/modelUnderwaters.mp4").subclipped(0, 0 + 5)\nclip2 = VideoFileClip("../Data/flipturnHQ.mp4").subclipped(0, 0 + 5)\n\n\ncombined = clips_array([[clip1], [clip2]])\n\neffect = vfx.Margin(left=960, right=960)\ncombined = effect.apply(combined)\neffect = vfx.Resize((1920,1080) )\ncombined = effect.apply(combined)\n\n#combined = combined.resize(height=1920)\n\n\neffect = vfx.Crop(x1=1166.6, y1=0, x2=2246.6, y2=1920)\n#clip = effect.apply(clip)\n#combined = combined.crop(x1=1166.6, y1=0, x2=2246.6, y2=1920)\n\n#combined = combined.with_effects(vfx.Resize((1920,1080)), vfx.Margin(960))\ncombin

In [2]:
import accelerate
from pathlib import Path

#print(accelerate.__version__)

c:\Users\danre\Documents\GitHub\SwimVisionProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:


import torch
import numpy as np
import cv2
import matplotlib
import argparse
import os
import time
from PIL import Image
from transformers import (
    AutoProcessor,
    RTDetrForObjectDetection,
    VitPoseForPoseEstimation,
)
print(torch.cuda.is_available())
device = 'cuda' if torch.cuda.is_available() else 'cpu'
index = 0
OUT_DIR = '../outputs'

while True:
    save =  'Trial - ' + str(index)
    outputPath = Path(f'{OUT_DIR}/{save}')
    if outputPath.is_dir():
        index += 1
        continue
    else:
        os.makedirs(f'{OUT_DIR}/{save}', exist_ok=False)
        OUT_DIR = OUT_DIR + f'/{save}'
        break


model_name = 'usyd-community/vitpose-plus-huge'



#save_name = 'Ivan'#args.input.split(os.path.sep)[-1].split('.')[0]
# Define codec and create VideoWriter object.

# Load detector.
person_image_processor = AutoProcessor.from_pretrained(
    'PekingU/rtdetr_r50vd_coco_o365'
)
person_model = RTDetrForObjectDetection.from_pretrained(
    'PekingU/rtdetr_r50vd_coco_o365', device_map=device
)
# Load ViTPose.
#print(f"Pose Model: {'../models/VitPose-s_RePoGen.pth'}")
image_processor = AutoProcessor.from_pretrained(model_name)
model = VitPoseForPoseEstimation.from_pretrained(pretrained_model_name_or_path=model_name, device_map=device)


#This is sadly chatted :(
# modelCustom = torch.load('../Models/VitPose-s_RePoGen.pth', device, weights_only=False)
# model.load_state_dict(modelCustom)
# image_processor.load_state_dict(modelCustom)
# image_processor.eval()
# model.eval()

True


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [4]:
import math


edges = [
    (0, 1), (0, 2), (2, 4), (1, 3), (6, 8), (8, 10),
    (5, 7), (7, 9), (5, 11), (11, 13), (13, 15), (6, 12),
    (12, 14), (14, 16), (5, 6), (11, 12)
]

def lowToHigh(inputList):
    if (inputList[0] < inputList[1]):
        return inputList
    else:
        return (inputList[1], inputList[0])

def detect_objects(image):
    """
    :param image: Image in PIL image format.
    Returns:
        person_boxes: Bboxes of persons in [x, y, w, h] format.
    """
    det_time_start = time.time()
    inputs = person_image_processor(
        images=image, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        outputs = person_model(**inputs)
    target_sizes = torch.tensor([(image.height, image.width)])
    
    results = person_image_processor.post_process_object_detection(
        outputs, target_sizes=target_sizes, threshold=0.3
    )
    det_time_end = time.time()
    det_fps = 1 / (det_time_end-det_time_start)
    
    # Extract the first result, as we can pass multiple images at a time.
    result = results[0]
    
    # In COCO dataset, humans labels have index 0.
    person_boxes_xyxy = result['boxes'][result['labels'] == 0]
    person_boxes_xyxy = person_boxes_xyxy.cpu().numpy()
    
    # Convert boxes from (x1, y1, x2, y2) to (x1, y1, w, h) format.
    person_boxes = person_boxes_xyxy.copy()
    person_boxes[:, 2] = person_boxes[:, 2] - person_boxes[:, 0]
    person_boxes[:, 3] = person_boxes[:, 3] - person_boxes[:, 1]
    return person_boxes, det_fps
def detect_pose(image, person_boxes):
    """
    :param image: Image in PIL image format.
    :param person_bboxes: Batched person boxes in [[x, y, w, h], ...] format.
    """
    pose_time_start = time.time()
    inputs = image_processor(
        image, boxes=[person_boxes], return_tensors='pt'
    ).to(device)
    
    dataset_index = torch.tensor([0], device=device) # must be a tensor of shape (batch_size,)
    if len(person_boxes) != 0:
        if 'plus' in model_name:
            with torch.no_grad():
                outputs = model(**inputs, dataset_index=dataset_index)
        else:
            with torch.no_grad():
                outputs = model(**inputs)
        
        pose_results = image_processor.post_process_pose_estimation(
            outputs, boxes=[person_boxes]
        )
    pose_time_end = time.time()
    pose_fps = 1 / (pose_time_end-pose_time_start)
    if len(person_boxes) == 0:
        return [], pose_fps
    image_pose_result = pose_results[0]
    #THIS IS A SANITY CHECK FOR THE MODEL

    realPoints = getKeypoints(image_pose_result)
    if (abs(slope(realPoints[5], realPoints[11]) > 2)): #More sanity check measures have to be made in the future if I don't get a better model
        image_pose_result = []
    return image_pose_result, pose_fps
def draw_keypoints(outputs, image):
    """
    :param outputs: Outputs from the keypoint detector.
    :param image: Image in PIL Image format.
    Returns:
        image: Annotated image Numpy array format.
    """
    image = cv2.cvtColor(np.array(image), cv2.COLOR_RGB2BGR)
    # the `outputs` is list which in-turn contains the dictionaries 
    for i, pose_result in enumerate(outputs):
        keypoints = pose_result['keypoints'].cpu().detach().numpy()
        # proceed to draw the lines if the confidence score is above 0.9
        keypoints = keypoints[:, :].reshape(-1, 2)
        # if keypoints.shape[0] != 0 and keypoints.shape[0] != 17:
        #     print(keypoints.shape[0]) 
        for p in range(keypoints.shape[0]):
            if p in [5, 7, 9, 11, 13, 15]:
                # draw the keypoints
                cv2.circle(image, (int(keypoints[p, 0]), int(keypoints[p, 1])), 
                            3, (0, 0, 255), thickness=-1, lineType=cv2.FILLED)
                # uncomment the following lines if you want to put keypoint number
                cv2.putText(image, f'{p}', (int(keypoints[p, 0]+10), int(keypoints[p, 1]-5)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)
            for ie, e in enumerate(edges):
                if ((set([5, 7, 9, 11, 13, 15]) & set(e)) and not 12 in e and not 6 in e):
                    # get different colors for the edges
                    rgb = matplotlib.colors.hsv_to_rgb([
                        ie/float(len(edges)), 1.0, 1.0
                    ])
                    rgb = rgb*255
                    # join the keypoint pairs to draw the skeletal structure
                    cv2.line(image, (int(keypoints[e, 0][0]), int(keypoints[e, 1][0])),
                            (int(keypoints[e, 0][1]), int(keypoints[e, 1][1])),
                            tuple(rgb), 1, lineType=cv2.LINE_AA)
        break
    return image

# def drawPersonBoxes(frame, boxes):
#     print(boxes)
#     for box in boxes:
#         cv2.rectangle(frame, (box[0], box[1]), (box[0] + box[2], box[1] + box[3]), (0,0,255), 2)
#     return frame

#Whole skeleton
# body_edges = [(6, 8), (8, 10),
#     (5, 7), (7, 9), (5, 11), (11, 13), (6, 12), (13, 15),
#     (12, 14), (14, 16), (5, 6), (11, 12)]

#Underwaters from left only
body_edges = [(5, 7), (7, 9), (5, 11), (11, 13), (13, 15)] #THESE NEED TO BE LOWER NUMBER FIRST FOR ANGLE OPERATIONS TO WORK


#https://stackoverflow.com/questions/14066933/direct-way-of-computing-the-clockwise-angle-between-two-vectors
# [left, vertex, right]
# Format needs to be this way because it calculates clockwise angle, not minimum angle
angles =[[11, 5, 7], [13, 11, 5], [11, 13, 15]] # Placing the reference vectors in the vertical may prove a liability


#angles =[[7, 5, 11], [5, 11, 13], [15, 13, 11]] 

def getAngles(points):
    angleList = []
    if points:
        for i in range(len(angles)):
            # print(points[angles[i][0]])
            # print(points[angles[i][1]])
            # print(points[angles[i][2]])

            x1 = points[angles[i][0]][0] - points[angles[i][1]][0]
            y1 = points[angles[i][0]][1] - points[angles[i][1]][1]
            x2 = points[angles[i][2]][0] - points[angles[i][1]][0]
            y2 = points[angles[i][2]][1] - points[angles[i][1]][1]
            # print(f'{x1}, {y1}    {x2}, {y2}')
            pointCombo = [[x1, y1], [x2, y2]]
            # print(pointCombo)
            # print(calcAngle(pointCombo))
            angleList.append(calcAngle(pointCombo))
    return angleList


def calcAngle(pointSet):
    p1, p2 = pointSet
    dot = (p1[0] * p2[0]) + (p1[1] * p2[1])
    det = (p1[0] * p2[1]) - (p1[1] * p2[0]) 
    rawAngle = math.atan2(-det, -dot) + math.pi 
    return 360- math.degrees(rawAngle)

#print(calcAngle([[-172, 29],[163, -34]]))

#print(calcAngle([[0.7, -.7], [-0.7, -0.7]]))


samplePoints = [(12, -34), #0
(-8, 41), #1
(25, 25), #2
(-47, -10), #3
(3, 19), #4
(33, -4), #5
(-21, -39), #6
(45, 12), #7
(-15, 28), #8
(0, -48), #9
(37, 37), #10
(-29, 5), #11
(18, -22), #12
(-1, -1), #13
(40, -40), #14
(-35, 15), #15
(10, 48)] #16

#getAngles(samplePoints)




def drawAngles(frame, angleSet, points):
    if points:
        #points = switchOpenCVNormal( image, points)
        # for point in points:
        #     print(point)
        for i in range(len(angleSet)):
            leftIndex = angles[i][0]
            vertexIndex = angles[i][1]
            rightIndex = angles[i][2]

            leftPoint = points[leftIndex]
            vertexPoint = points[vertexIndex]
            rightPoint = points[rightIndex]

            normalizedLeftPoint = [leftPoint[0]- vertexPoint[0], leftPoint[1]-vertexPoint[1]]
            normalizedRightPoint = [rightPoint[0]- vertexPoint[0], rightPoint[1]-vertexPoint[1]]
            extraPointSet = [normalizedLeftPoint, [1,0]]
            textAngle = (angleSet[i]/2) + calcAngle(extraPointSet)
            ellipseStartAngle = calcAngle(extraPointSet)
            ellipseEndAngle = calcAngle(extraPointSet) + angleSet[i] 

            cv2.ellipse(frame, (vertexPoint[0], vertexPoint[1]), (30, 30), 0, ellipseStartAngle, ellipseEndAngle, (0,0,255), 2)

            # print(f"Three points: {leftPoint}, {vertexPoint}, {rightPoint}")
            # print(f"Initial Angle: {angleSet[i]}")
            # print(f"Absolute Angle: {absoluteAngle}")

            drawPoint = [vertexPoint[0] + math.cos(math.radians(textAngle))*70 -20, vertexPoint[1] + math.sin(math.radians(textAngle))*70]

            cv2.putText(frame, f'{round(angleSet[i], 2)}', (int(drawPoint[0]), int(drawPoint[1])), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (240, 19, 255), 2) # I converted to left-hand coordinates here
    return frame



def drawAngleDiffs(frame, angleSet, diffAngleSet, points):
    if points and diffAngleSet:
        #points = switchOpenCVNormal( image, points)
        # for point in points:
        #     print(point)

        for i in range(len(angleSet)):
            leftIndex = angles[i][0]
            vertexIndex = angles[i][1]
            rightIndex = angles[i][2]

            leftPoint = points[leftIndex]
            vertexPoint = points[vertexIndex]
            rightPoint = points[rightIndex]

            normalizedLeftPoint = [leftPoint[0]- vertexPoint[0], leftPoint[1]-vertexPoint[1]]
            normalizedRightPoint = [rightPoint[0]- vertexPoint[0], rightPoint[1]-vertexPoint[1]]
            extraPointSet = [normalizedLeftPoint, [1,0]]
            textAngle = (angleSet[i]/2) + calcAngle(extraPointSet)
            ellipseStartAngle = calcAngle(extraPointSet)
            ellipseEndAngle = calcAngle(extraPointSet) + angleSet[i] 

            cv2.ellipse(frame, (vertexPoint[0], vertexPoint[1]), (30, 30), 0, ellipseStartAngle, ellipseEndAngle, (0,0,255), 2)

            # print(f"Three points: {leftPoint}, {vertexPoint}, {rightPoint}")
            # print(f"Initial Angle: {angleSet[i]}")
            # print(f"Absolute Angle: {absoluteAngle}")
            yOffset = 0
            
            anglePoint = [vertexPoint[0] + math.cos(math.radians(textAngle))*70 -20, vertexPoint[1] + math.sin(math.radians(textAngle))*70]
            if (math.sin(math.radians(textAngle)) > 0):
                yOffset = 30
            else:
                yOffset = -30
            cv2.putText(frame, f'{round(angleSet[i], 2)}', (int(anglePoint[0]), int(anglePoint[1])), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (240, 19, 255), 2)
            if (diffAngleSet[i] < 0):
                cv2.putText(frame, f'({round(diffAngleSet[i], 2)})', (int(anglePoint[0])+10, int(anglePoint[1]) + yOffset), cv2.FONT_HERSHEY_SIMPLEX, 0.59, (23, 122, 5), 2)
            else:
                cv2.putText(frame, f'({round(diffAngleSet[i], 2)})', (int(anglePoint[0])+10, int(anglePoint[1]) + yOffset), cv2.FONT_HERSHEY_SIMPLEX, 0.59, (0, 0, 255), 2)
    return frame

pointOfFixation = 11

def cacheOffsets(points):
    xDistance = abs(points[15][0] - points[9][0])
    return [[abs(points[pointOfFixation][0]-points[9][0]), abs(points[15][0] - points[pointOfFixation][0])], [int(xDistance/5), int(xDistance/5)]]



def cropImage(frame, points, offsets, frameHeight, frameWidth):
    # frame
    # lowestX = points[0][0]
    # highestX = points[0][0]
    # lowestY = points[0][1]
    # highestY = points[0][1]
    # for point in points:
    #     lowestX = min(lowestX, point[0])
    #     lowestY = min(lowestY, point[1])
    #     highestX = max(highestX, point[0])
    #     highestY = max(highestY, point[1])
    buffer = 200

    lowX = points[pointOfFixation][0] - offsets[0][0]
    highX = points[pointOfFixation][0] + offsets[0][1]
    lowY = points[pointOfFixation][1] - offsets[1][0]
    highY = points[pointOfFixation][1] + offsets[1][1]

    origin = (max(lowX - buffer, 0), max(lowY - int(buffer/3), 0))
    terminal = (min(highX + buffer, frameWidth), min(highY + int(buffer/1.5), frameHeight))

    points = [origin, terminal]
    croppedImage = frame[origin[1]:terminal[1], origin[0]: terminal[0]]
    croppedImage = cv2.resize(croppedImage, (frameWidth, frameHeight), interpolation=cv2.INTER_CUBIC)
    return croppedImage
        


def getKeypoints(outputs):
    out = []
    if outputs:
        keypoints = outputs[0]['keypoints'].cpu().detach().numpy()
        keypoints = keypoints[:, :].reshape(-1, 2)
        for p in range(keypoints.shape[0]):
            out.append([int(keypoints[p, 0]), int(keypoints[p, 1])])
    return out

def drawSlopes(image, slopes, points):
    if slopes and points:
        for i, edge in enumerate(body_edges):
            average = getAverageCoord(points[edge[0]], points[edge[1]])
            cv2.putText(image, f'{round(slopes[i], 2)}', (average[0] -10, average[1]),
                             cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)           
    return image



def getSlopes(outputs, image, points):
    slopes = []
    if outputs and points:
        for i, edge in enumerate(body_edges):
            slopes.append(slope(points[edge[0]], points[edge[1]]))
    return slopes


def switchOpenCVNormal(image, kps):
    newkps = copy.deepcopy(kps)
    #print(newkps == kps)
    width, height = image.size
    for i in range(len(newkps)):
        newkps[i][1] = height - newkps[i][1]
    return newkps

def getAverageCoord(coord1, coord2):
    return [int((coord1[0] + coord2[0])/2), int((coord1[1] + coord2[1])/2)]

def slope(coord1, coord2):
    if coord1[0] - coord2[0] != 0:
        return ((coord1[1] - coord2[1]) / (coord1[0] - coord2[0])) 
    else:
        return 100 #Arbitrarily high slope


In [5]:

def flipNeededAndOffsets(videoName):
    inputPath = '../Data/' + videoName + '.mp4'
    cap = cv2.VideoCapture(inputPath)
    vidLength = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if not cap.isOpened():
        print("Not opening")
    curFrame = int(vidLength/2)
    output = False
    offsets = [[2000, 2000], [2000, 2000]]
    while True:
        cap.set(cv2.CAP_PROP_POS_FRAMES, curFrame)
        res, frame = cap.read()
        if res:
            #cv2.imshow('Frame', frame)
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(frame_rgb)
            bboxes, det_fps = detect_objects(image=image)
            image_pose_result, pose_fps = detect_pose(image=image, person_boxes=bboxes)
            kps = getKeypoints(image_pose_result)
            normalkps = switchOpenCVNormal(image, kps)
            if normalkps:
                output = normalkps[5][0] > normalkps[11][0]
                offsets = cacheOffsets(kps)
                break
        curFrame = curFrame+1
        if curFrame == vidLength-1:
            break
    cap.release()
    return [output, offsets]
            
    


        




def makeMainVideo(videoName, flipNeeded, offsets):
    inputPath = '../Data/' + videoName + '.mp4'
    cap = cv2.VideoCapture(inputPath)
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    video_fps = int(cap.get(5)) 
    
    frame_count = 0 # To count total frames.
    total_fps = 0 # To get the final frames per second.
    masterAngleList = []

    # offsets = [[2000, 2000], [2000, 2000]]
    # cachedOffsets = False
    cachedkps = [0,0] * 17

    os.makedirs(f'{OUT_DIR}/{videoName}', exist_ok=False)
    out = cv2.VideoWriter(
        f'{OUT_DIR}/{videoName}/MainVideo.mp4', 
        cv2.VideoWriter_fourcc(*'avc1'), 
        video_fps, 
        (frame_width, frame_height), 
    )



    if not cap.isOpened():
        print("Not opening")
        
    while cap.isOpened():
        ret, frame = cap.read()
        if ret:
            if flipNeeded:
                frame = cv2.flip(frame, 1)
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            
            image = Image.fromarray(frame_rgb)

            start_time = time.time()
            bboxes, det_fps = detect_objects(image=image)
            image_pose_result, pose_fps = detect_pose(image=image, person_boxes=bboxes)
            result = draw_keypoints(image_pose_result, image)
            kps = getKeypoints(image_pose_result)
            normalkps = switchOpenCVNormal(image, kps)
            #listSlopes = getSlopes(image_pose_result, image, normalkps)
            #result = drawSlopes(result, listSlopes, kps)
            
            angle = getAngles(normalkps)
            result = drawAngles(result, angle, kps)

            # if not cachedOffsets:
            #     if kps:
            #         newOffsets = cacheOffsets(kps)
            #         offsets[0][0] = newOffsets[0][0]
            #         offsets[0][1] = newOffsets[0][1]
            #         offsets[1][0] = newOffsets[1][0]
            #         offsets[1][1] = newOffsets[1][1]
            #         cachedOffsets = True
        
            if kps:
                cachedkps = kps

            result = cropImage(result, cachedkps, offsets, frame_height, frame_width)
            
            masterAngleList.append(angle)
            end_time = time.time()
            forward_pass_time = end_time - start_time
                
            # Get the current fps.
            fps = 1 / (forward_pass_time)
            # Add `fps` to `total_fps`.
            total_fps += fps
            # Increment frame count.
            frame_count += 1
            '''cv2.putText(
                result,
                f"Frame Count: {frame_count-1} | FPS: {fps:0.1f} | Pose FPS: {pose_fps:0.1f} | Detection FPS: {det_fps:0.1f}",
                (15, 25),
                fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                fontScale=1.0,
                color=(0, 0, 255),
                thickness=2,
                lineType=cv2.LINE_AA,
            )'''
            out.write(result)
            cv2.imshow('Prediction', result)
            # Press `q` to exit
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        else:
            break
    cap.release()
    cv2.destroyAllWindows()
    out.release()
    avg_fps = total_fps/frame_count
    print(f"Average FPS: {avg_fps}")
    return masterAngleList



 # Release VideoCapture().

# Close all frames and video windows.
def findFirstFrame(angles):
    for i in range(len(angles)):
        if (angles[i]):
            if (angles[i][1] > 155 and angles[i][2]<145):
                return i
    return len(angles) -1 


In [6]:
# print(masterSlopeList)
# print(len(masterSlopeList))
# print(frame_count)

In [7]:


targetAngleID = 2

def findSmallest(angleList, initial):
    if angleList[initial]:
        lastAngle = angleList[initial][targetAngleID]
    for i in range (initial, len(angleList)):
        print(f"CHECKING FRAME {i}")
        if angleList[i]:
            curAngle = angleList[i][targetAngleID]
            print(curAngle)
            if (curAngle - lastAngle > 1):
                return i 
            lastAngle = curAngle
            end = i
    return end-1

def findLargest(angleList, initial):
    if angleList[initial]:
        lastAngle = angleList[initial][targetAngleID]
    largestFrame = 0
    for i in range (initial, len(angleList)):
        print(f"CHECKING FRAME {i}")
        if angleList[i]:
            curAngle = angleList[i][targetAngleID]
            print(curAngle)
            if (curAngle > angleList[largestFrame][targetAngleID]):
                largestFrame = i
            if (lastAngle - curAngle > 2):
                return [i, largestFrame] 
            lastAngle = curAngle
            end = i
    return [end-1, end-1]


def makeClips(videoName, angleList, flipNeeded):
    kickFrames = []
    start = findFirstFrame(angleList)
    print(f"STARTING FRAME: {start}")
    i = 0
    final = -1
    while True:
        if i == 0:
            initial = findSmallest(angleList, start) -1
            print(initial)
        else:
            initial = start
        middle = findLargest(angleList, initial)
        print(middle)
        final = findSmallest(angleList, middle[0])
        print(f'{[initial, middle, final-1]}')
        if initial != final-1 and angleList[final][2] < 155: #Rule 2 that may be needed: middle != final-1 
            kickFrames.append([initial, middle[1]-1, final-1])
        else:
            break 
        start = final-1 
        i = i+1

    print(kickFrames)

    # fileName = 'ShortUnderwaters.mp4'
    # inputPath = '../Data/' + fileName
    # cap = cv2.VideoCapture(inputPath)
    os.makedirs(f'{OUT_DIR}/{videoName}/Clips')
    for frameCombo in kickFrames:
        inputPath = '../Data/' + videoName + '.mp4'
        cap2 = cv2.VideoCapture(inputPath)
        #cap2 = cv2.VideoCapture(f'{OUT_DIR}/{videoName}/MainVideo.mp4') # OLD WITH COLORS
        if not cap2.isOpened():
            print("Error: Could not open video file.")
        else:
            print("Video file opened successfully!")
        index = 0
        frame_width = int(cap2.get(3))
        frame_height = int(cap2.get(4))
        video_fps = int(cap2.get(5)) 
        while True:
            clip_save_index = str(index)
            outputPath = Path(f'{OUT_DIR}/{videoName}/Clips/{clip_save_index}.mp4')
            if outputPath.is_file():
                index += 1
                continue
            else:
                out2 = cv2.VideoWriter(
                    f'{OUT_DIR}/{videoName}/Clips/{clip_save_index}.mp4', 
                    cv2.VideoWriter_fourcc(*'avc1'), 
                    video_fps, 
                    (frame_width, frame_height), 
                )
                index = index + 1
                break
        
        for i in range(frameCombo[0], frameCombo[2]):
            cap2.set(cv2.CAP_PROP_POS_FRAMES, i)
            res, frame = cap2.read()
            if res:
                if flipNeeded:
                    frame = cv2.flip(frame, 1)
                #cv2.imshow('Frame', frame)
                out2.write(frame)
        #print(f'{OUT_DIR}/clipOutputs/{save_name}.mp4')
        cap2.release()
        out2.release()
        cv2.destroyAllWindows()
    return kickFrames

    
    

# Close all frames and video windows.

 

Agenda for tmrw:

1. Throw the slope gathering into a method (done)
2. Make clip creation a method (done)
3. pick minimum amount of kicks and compare the first n kicks (done)
4. Create clips, make sure the learner clips are normalized to the expert's length (doing)
5. Calc difference every single slope between each frame of the normalized clips.  Store best and worse absolute diff + average for that frame
6. Create a list for each frame that records diff of each edge
7. Make a method to turn  diff list into summary list with 
 [average, [best joint, accuracy], [worst frame, accuracy]]
8. Make methods to extract this data to find overall superlatives (best frame, worst frame, worst joint of them all, best joint of them all)
9. Get accuracy for each kick and overall
10. If you really want splice the kick clips tg, get acc % differences for each joint every frame of recording, color grade them based on % diff, extend clip to like 10 seconds 

In [8]:

def getClipAngles(videoName, number, learner):
    if learner:
        inputPath = f'{OUT_DIR}/{videoName}/Clips/{number}N.mp4'
    else:
        inputPath = f'{OUT_DIR}/{videoName}/Clips/{number}.mp4'
    cap = cv2.VideoCapture(inputPath)
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    video_fps = int(cap.get(5)) 

    # out = cv2.VideoWriter(
    #     f'{OUT_DIR}/{videoName}/Clips/{number}NA.mp4', 
    #     cv2.VideoWriter_fourcc(*'avc1'), 
    #     video_fps, 
    #     (frame_width, frame_height), 
    # )
    
    frame_count = 0 # To count total frames.
    total_fps = 0 # To get the final frames per second.
    angleList = []
    if not cap.isOpened():
        print("Not opening")
        
    while cap.isOpened():
        ret, frame = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = Image.fromarray(frame_rgb)
            start_time = time.time()
            bboxes, det_fps = detect_objects(image=image)
            image_pose_result, pose_fps = detect_pose(image=image, person_boxes=bboxes)
            result = draw_keypoints(image_pose_result, image)
            kps = getKeypoints(image_pose_result)
            normalkps = switchOpenCVNormal(image, kps)
            #listSlopes = getSlopes(image_pose_result, image, normalkps)
            #result = drawSlopes(result, listSlopes, kps)
            angle = getAngles(normalkps)
            angleList.append(angle)
            #result = drawAngles(result, angle, kps)
            end_time = time.time()
            forward_pass_time = end_time - start_time
                
            # Get the current fps.
            fps = 1 / (forward_pass_time)
            # Add `fps` to `total_fps`.
            total_fps += fps
            # Increment frame count.
            frame_count += 1
            '''cv2.putText(
                result,
                f"Frame Count: {frame_count-1} | FPS: {fps:0.1f} | Pose FPS: {pose_fps:0.1f} | Detection FPS: {det_fps:0.1f}",
                (15, 25),
                fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                fontScale=1.0,
                color=(0, 0, 255),
                thickness=2,
                lineType=cv2.LINE_AA,
            )'''
            #out.write(result)
            #cv2.imshow('Prediction', result)
            # Press `q` to exit
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        else:
            break
    cap.release()
    cv2.destroyAllWindows()
    #out.release()
    avg_fps = total_fps/frame_count
    print(f"Average FPS: {avg_fps}")
    return angleList


        


def normalizeLearnersClips(learnersVid, expertsVid, usableKicks):

    for i in range(usableKicks):
        capExpert = cv2.VideoCapture(f'{OUT_DIR}/{expertsVid}/Clips/{i}.mp4')
        targetLength = int(capExpert.get(cv2.CAP_PROP_FRAME_COUNT))
        print(targetLength)
        capExpert.release()
        capLearner = cv2.VideoCapture(f'{OUT_DIR}/{learnersVid}/Clips/{i}.mp4')
        currentLength = int(capLearner.get(cv2.CAP_PROP_FRAME_COUNT))
        if targetLength - currentLength == 0:
            interval = 0
        else:
            interval = currentLength / (targetLength - currentLength)

        print(f'\nTarget length: {targetLength}\nCurrentLength: {currentLength}\nInterval: {interval}\n')
        frameCount = 0
        frame_width = int(capLearner.get(3))
        frame_height = int(capLearner.get(4))
        video_fps = int(capLearner.get(5))
        count = interval
        out3 = cv2.VideoWriter(
            f'{OUT_DIR}/{learnersVid}/Clips/{i}N.mp4', 
            cv2.VideoWriter_fourcc(*'avc1'), 
            video_fps, 
            (frame_width, frame_height), 
        )
        while capLearner.isOpened():
            ret, frame = capLearner.read()
            if ret:
                frameCount = frameCount + 1
                if interval == 0:
                    out3.write(frame)
                    print(f"Frame {frameCount}: normal frame")
                else: 

                    print(f'Count: {count}, Rounded: {round(count)}, frameCount: {frameCount}, interval: {interval}, ')
                    if (round(count) == frameCount):
                        out3.write(frame) 
                        out3.write(frame)
                        print(f"Frame {frameCount}: double Frame")
                        count = count + interval
                    elif (-1*(round(count)) == frameCount):
                        print(f"Frame {frameCount}: frame Skipped")
                        count = count + interval
                    else:
                        print(f"Frame {frameCount}: normal frame")
                        out3.write(frame)
                    # if round(frameCount * interval) == : #Double frame
                    #     out3.write(frame) 
                    #     out3.write(frame)
                    #     print(f"Frame {frameCount}: double Frame")
                    # elif frameCount % (-1*interval) == 0:
                    #     print(f"Frame {frameCount}: frame Skipped")
                    # else:
                    #     print(f"Frame {frameCount}: normal frame")
                    #     out3.write(frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
            else:
                break
        capLearner.release()
        out3.release()
        cv2.destroyAllWindows()
        print("all done!")
        # del capLearner
        # del out3



def revertLearnersClips(learnersVid, usableKicks):
    #assuming we get least amount of kicks somewhere
    for i in range(usableKicks):
        capOriginal = cv2.VideoCapture(f'{OUT_DIR}/{learnersVid}/Clips/{i}.mp4')
        targetLength = int(capOriginal.get(cv2.CAP_PROP_FRAME_COUNT))
        print(targetLength)
        capOriginal.release()
        capNormalized = cv2.VideoCapture(f'{OUT_DIR}/{learnersVid}/Clips/{i}NA.mp4')
        currentLength = int(capNormalized.get(cv2.CAP_PROP_FRAME_COUNT))
        if targetLength - currentLength == 0:
            interval = 0
        else:
            interval = currentLength / (targetLength - currentLength)

        print(f'\nTarget length: {targetLength}\nCurrentLength: {currentLength}\nInterval: {interval}\n')
        frameCount = 0
        frame_width = int(capNormalized.get(3))
        frame_height = int(capNormalized.get(4))
        video_fps = int(capNormalized.get(5))
        count = interval
        out3 = cv2.VideoWriter(
            f'{OUT_DIR}/{learnersVid}/Clips/{i}A.mp4', 
            cv2.VideoWriter_fourcc(*'avc1'), 
            video_fps, 
            (frame_width, frame_height), 
        )
        while capNormalized.isOpened():
            ret, frame = capNormalized.read()
            if ret:
                frameCount = frameCount + 1
                if interval == 0:
                    out3.write(frame)
                    print(f"Frame {frameCount}: normal frame")
                else: 

                    print(f'Count: {count}, Rounded: {round(count)}, frameCount: {frameCount}, interval: {interval}, ')
                    if (round(count) == frameCount):
                        out3.write(frame) 
                        out3.write(frame)
                        print(f"Frame {frameCount}: double Frame")
                        count = count + interval
                    elif (-1*(round(count)) == frameCount):
                        print(f"Frame {frameCount}: frame Skipped")
                        count = count + interval
                    else:
                        print(f"Frame {frameCount}: normal frame")
                        out3.write(frame)
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
            else:
                break
        capNormalized.release()
        out3.release()
        cv2.destroyAllWindows()
        print("all done!")
        # del capLearner
        # del out3


def compareClip(learnersVid, expertsVid, kicks):
    normalizeLearnersClips(learnersVid, expertsVid, kicks)
    masterlist = []
    #Starting the clip access phase
    for k in range(kicks):
        diffAnglesList = []
        learnerAngles = getClipAngles(learnersVid, k, True)
        expertAngles = getClipAngles(expertsVid, k, False)
        outputList = []
        for i in range(len(learnerAngles)):
            miniDiffsList = []
            if learnerAngles[i] and expertAngles[i]:
                for j in range(len(learnerAngles[i])): #I think it's just 17
                    diff = learnerAngles[i][j] - expertAngles[i][j]
                    miniDiffsList.append(diff)
                    outputList.append([f"F{i} E{j}", diff])
            diffAnglesList.append(miniDiffsList)
                
        sorted(outputList, key=lambda x: (x[1])) #THIS IS THE ERROR CULPRIT
        for elem in outputList:
            print(f'{elem[0]}: {elem[1]}')
        masterlist.append(diffAnglesList)
    return masterlist

#compareClip("OtherUnderwater", "ShortUnderwater", 0)

#normalizeLearnersClips("OtherUnderwater", "ShortUnderwater")



def annotateNormalizedVideos(learnersVid, diffList, usableKicks, offsets):
    for i in range(usableKicks):
        cap = cv2.VideoCapture(f'{OUT_DIR}/{learnersVid}/Clips/{i}N.mp4')
        frame_width = int(cap.get(3))
        frame_height = int(cap.get(4))
        video_fps = int(cap.get(5))
        out = cv2.VideoWriter(
            f'{OUT_DIR}/{learnersVid}/Clips/{i}NA.mp4', 
            cv2.VideoWriter_fourcc(*'avc1'), 
            video_fps, 
            (frame_width, frame_height), 
        )
        frame_count = 0

        
        cachedkps = [0,0] * 17

        if not cap.isOpened():
            print("Not opening")
        while cap.isOpened():
            ret, frame = cap.read()
            if ret:
                #frame = cv2.flip(frame, 1)
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                image = Image.fromarray(frame_rgb)
                start_time = time.time()
                bboxes, det_fps = detect_objects(image=image)
                image_pose_result, pose_fps = detect_pose(image=image, person_boxes=bboxes)
                result = draw_keypoints(image_pose_result, image)
                kps = getKeypoints(image_pose_result)
                normalkps = switchOpenCVNormal(image, kps)
                #listSlopes = getSlopes(image_pose_result, image, normalkps)
                #result = drawSlopes(result, listSlopes, kps)
                angle = getAngles(normalkps)
                result = drawAngleDiffs(result, angle, diffList[i][frame_count], kps)
                end_time = time.time()
                # forward_pass_time = end_time - start_time

                # if not cachedOffsets:
                #     if kps:
                #         newOffsets = cacheOffsets(kps)
                #         offsets[0][0] = newOffsets[0][0]
                #         offsets[0][1] = newOffsets[0][1]
                #         offsets[1][0] = newOffsets[1][0]
                #         offsets[1][1] = newOffsets[1][1]
                #         cachedOffsets = True
            
                if kps:
                    cachedkps = kps

                result = cropImage(result, cachedkps, offsets, frame_height, frame_width)

                # Get the current fps.
                #fps = 1 / (forward_pass_time)
                # Add `fps` to `total_fps`.
                #total_fps += fps
                # Increment frame count.
                frame_count += 1
                '''cv2.putText(
                    result,
                    f"Frame Count: {frame_count-1} | FPS: {fps:0.1f} | Pose FPS: {pose_fps:0.1f} | Detection FPS: {det_fps:0.1f}",
                    (15, 25),
                    fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                    fontScale=1.0,
                    color=(0, 0, 255),
                    thickness=2,
                    lineType=cv2.LINE_AA,
                )'''
                out.write(result)
                #cv2.imshow('Prediction', result)
                # Press `q` to exit
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
            else:
                break
        cap.release()
        cv2.destroyAllWindows()
        out.release()
        #avg_fps = total_fps/frame_count
        #print(f"Average FPS: {avg_fps}")
        
    







What is left:

1. Fill in gaps between kicks (done)
2. Make final display video
3. Make rules that figure out when underwater starts + how many kicks there are (done)
4. Similarity report
5. Possible stretches given low angle for smth

In [9]:
# cap2 = cv2.VideoCapture(f"{OUT_DIR}/OtherUnderwater/Clips/0NA.mp4")
# print(int(cap2.get(cv2.CAP_PROP_FRAME_COUNT)))
# cap2.release()
# del cap2
# cap2 = cv2.VideoCapture(f"{OUT_DIR}/OtherUnderwater/Clips/0.mp4")
# print(int(cap2.get(cv2.CAP_PROP_FRAME_COUNT)))
# cap2.release()
# del cap2
# cap2 = cv2.VideoCapture(f"{OUT_DIR}/OtherUnderwater/Clips/0A.mp4")
# print(int(cap2.get(cv2.CAP_PROP_FRAME_COUNT)))
# cap2.release()

In [10]:

#print(MainAngles1)

In [11]:
#MainAngles3 = makeMainVideo("flipTurnHQ")

def makePreComparisonClip(vidName, kickFrames):
    cap = cv2.VideoCapture(f'{OUT_DIR}/{vidName}/MainVideo.mp4')
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    video_fps = int(cap.get(5))
    out = cv2.VideoWriter(
        f'{OUT_DIR}/{vidName}/Clips/PreDetectionClip.mp4', 
        cv2.VideoWriter_fourcc(*'avc1'), 
        video_fps, 
        (frame_width, frame_height), 
    )
    for i in range(0, kickFrames[0][0]):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        res, frame = cap.read()
        if res:
            #cv2.imshow('Frame', frame)
            out.write(frame)
    cap.release()
    out.release()

def makePostComparisonClip(vidName, kickFrames, usableKicks):
    cap = cv2.VideoCapture(f'{OUT_DIR}/{vidName}/MainVideo.mp4')
    vidLength = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    video_fps = int(cap.get(5))
    out = cv2.VideoWriter(
        f'{OUT_DIR}/{vidName}/Clips/PostDetectionClip.mp4', 
        cv2.VideoWriter_fourcc(*'avc1'), 
        video_fps, 
        (frame_width, frame_height), 
    )
    for i in range(kickFrames[usableKicks-1][2], vidLength):
        cap.set(cv2.CAP_PROP_POS_FRAMES, i)
        res, frame = cap.read()
        if res:
            #cv2.imshow('Frame', frame)
            out.write(frame)
    cap.release()
    out.release()

def makeOutputVid1(vidName, kickFrames, kicks):
    makePreComparisonClip(vidName, kickFrames)
    cap = cv2.VideoCapture(f'{OUT_DIR}/{vidName}/Clips/PreDetectionClip.mp4')
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    video_fps = int(cap.get(5))
    out = cv2.VideoWriter(
        f'{OUT_DIR}/{vidName}/FinalVideo1.mp4', 
        cv2.VideoWriter_fourcc(*'avc1'), 
        video_fps, 
        (frame_width, frame_height), 
    )
    if not cap.isOpened():
            print("Not opening")
    while cap.isOpened():
        ret, frame = cap.read()
        if ret:
            #I can add annotations here
            out.write(frame)
            # Press `q` to exit
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        else:
            break
    cap.release()
    del cap
    for i in range(kicks):
        cap = cv2.VideoCapture(f'{OUT_DIR}/{vidName}/Clips/{i}A.mp4')
        if not cap.isOpened():
            print("Not opening")
        while cap.isOpened():
            ret, frame = cap.read()
            if ret:
                #I can add annotations here
                out.write(frame)
                # out.write(frame)
                # out.write(frame)
                # out.write(frame)
                # Press `q` to exit
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
            else:
                break
        cap.release()
        del cap
    makePostComparisonClip(vidName, kickFrames, kicks)
    cap = cv2.VideoCapture(f'{OUT_DIR}/{vidName}/Clips/PostDetectionClip.mp4')
    if not cap.isOpened():
        print("Not opening")
    while cap.isOpened():
        ret, frame = cap.read()
        if ret:
            #I can add annotations here
            out.write(frame)
            # Press `q` to exit
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        else:
            break
    cap.release()
    del cap
    out.release()

def makeOutputVid2(learnerVid, kickFrames, usableKicks):
    print(kickFrames)
    cap = cv2.VideoCapture(f'{OUT_DIR}/{learnerVid}/FinalVideo1.mp4')
    #vidLength = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    video_fps = int(cap.get(5))
    out = cv2.VideoWriter(
        f'{OUT_DIR}/{learnerVid}/FinalVideo2.mp4', 
        cv2.VideoWriter_fourcc(*'avc1'), 
        video_fps, 
        (frame_width, frame_height), 
    )
    for i in range(usableKicks):
        #print(i)
        cap.set(cv2.CAP_PROP_POS_FRAMES, kickFrames[i][0])
        res, frame = cap.read()
        if res:
            cv2.putText(
                frame,
                f"Kick {i+1} High",
                (40, 40),
                fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                fontScale=1.5,
                color=(0, 0, 255),
                thickness=2,
                lineType=cv2.LINE_AA,
            )
            for j in range(video_fps*3):
                out.write(frame)
                
        #print(kickFrames[i])
        cap.set(cv2.CAP_PROP_POS_FRAMES, kickFrames[i][1])
        res, frame = cap.read()
        if res:
            cv2.putText(
                frame,
                f"Kick {i+1} Low",
                (40, 40),
                fontFace=cv2.FONT_HERSHEY_SIMPLEX,
                fontScale=1.5,
                color=(0, 0, 255),
                thickness=2,
                lineType=cv2.LINE_AA,
            )
            for j in range(video_fps*3):
                out.write(frame)
    cap.release()
    out.release()


def synthesizeNormalizedLearnersClips(learnerVid, usableKicks):
    for i in range(usableKicks):
        cap = cv2.VideoCapture(f'{OUT_DIR}/{learnerVid}/Clips/{i}NA.mp4')
        if i == 0:
            frame_width = int(cap.get(3))
            frame_height = int(cap.get(4))
            video_fps = int(cap.get(5))
            out = cv2.VideoWriter(
                f'{OUT_DIR}/{learnerVid}/Clips/SynthesizedClips.mp4', 
                cv2.VideoWriter_fourcc(*'avc1'), 
                video_fps, 
                (frame_width, frame_height), 
            )
        if not cap.isOpened():
            print("Not opening")
        while cap.isOpened():
            ret, frame = cap.read()
            if ret:
                #I can add annotations here
                out.write(frame)
                # Press `q` to exit
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
            else:
                break
        cap.release()
        del cap
    out.release()

def makeAnnotatedExpertClips(expertVid, kickFrames, usableKicks):
    
    for i in range(usableKicks):
        cap = cv2.VideoCapture(f'{OUT_DIR}/{expertVid}/MainVideo.mp4')
        frame_width = int(cap.get(3))
        frame_height = int(cap.get(4))
        video_fps = int(cap.get(5))
        out = cv2.VideoWriter(
            f'{OUT_DIR}/{expertVid}/Clips/{i}A.mp4', 
            cv2.VideoWriter_fourcc(*'avc1'), 
            video_fps, 
            (frame_width, frame_height), 
        )
        for j in range(kickFrames[i][0], kickFrames[i][2]):
            cap.set(cv2.CAP_PROP_POS_FRAMES, j)
            ret, frame = cap.read()
            if ret:
                #I can add annotations here
                out.write(frame)
                # Press `q` to exit
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
            else:
                break
        cap.release()
        out.release()


    


def synthesizeExpertsClips(expertVid, usableKicks):
    for i in range(usableKicks):
        cap = cv2.VideoCapture(f'{OUT_DIR}/{expertVid}/Clips/{i}A.mp4')
        if i == 0:
            frame_width = int(cap.get(3))
            frame_height = int(cap.get(4))
            video_fps = int(cap.get(5))
            out = cv2.VideoWriter(
                f'{OUT_DIR}/{expertVid}/Clips/SynthesizedClips.mp4', 
                cv2.VideoWriter_fourcc(*'avc1'), 
                video_fps, 
                (frame_width, frame_height), 
            )
        if not cap.isOpened():
            print("Not opening")
        while cap.isOpened():
            ret, frame = cap.read()
            if ret:
                #I can add annotations here
                out.write(frame)
                # Press `q` to exit
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
            else:
                break
        cap.release()
        del cap
    out.release()


def makeOutputVid3(expertVid, learnerVid, kickFrames2, usableKicks):
    synthesizeNormalizedLearnersClips(learnerVid, usableKicks)
    makeAnnotatedExpertClips(expertVid, kickFrames2, usableKicks)
    synthesizeExpertsClips(expertVid, usableKicks)

    synthesizedLearners = VideoFileClip(f'{OUT_DIR}/{learnerVid}/Clips/SynthesizedClips.mp4')
    synthesizedExperts = VideoFileClip(f'{OUT_DIR}/{expertVid}/Clips/SynthesizedClips.mp4')
    

    combined = clips_array([[synthesizedLearners], [synthesizedExperts]])
    effect = vfx.Margin(left=960, right=960)
    combined = effect.apply(combined)
    effect = vfx.Resize((1920,1080) )
    combined = effect.apply(combined)

    #combined = combined.resize(height=1920)

    effect = vfx.Crop(x1=1166.6, y1=0, x2=2246.6, y2=1920)
    #clip = effect.apply(clip)
    #combined = combined.crop(x1=1166.6, y1=0, x2=2246.6, y2=1920)

    #combined = combined.with_effects(vfx.Resize((1920,1080)), vfx.Margin(960))
    combined.display_in_notebook
    #combined.write_videofile('test.mp4')

    combined.write_videofile(f'{OUT_DIR}/{learnerVid}/FinalVideo3.mp4')



def makeOutputVid4(vidName, kickFrames, kicks):
    makePreComparisonClip(vidName, kickFrames)
    cap = cv2.VideoCapture(f'{OUT_DIR}/{vidName}/Clips/PreDetectionClip.mp4')
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    video_fps = int(cap.get(5))
    out = cv2.VideoWriter(
        f'{OUT_DIR}/{vidName}/FinalVideo4.mp4', 
        cv2.VideoWriter_fourcc(*'avc1'), 
        video_fps, 
        (frame_width, frame_height), 
    )
    if not cap.isOpened():
            print("Not opening")
    while cap.isOpened():
        ret, frame = cap.read()
        if ret:
            #I can add annotations here
            out.write(frame)
            # Press `q` to exit
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        else:
            break
    cap.release()
    del cap
    for i in range(kicks):
        cap = cv2.VideoCapture(f'{OUT_DIR}/{vidName}/Clips/{i}A.mp4')
        if not cap.isOpened():
            print("Not opening")
        while cap.isOpened():
            ret, frame = cap.read()
            if ret:
                #I can add annotations here
                out.write(frame)
                out.write(frame)
                out.write(frame)
                out.write(frame)
                # Press `q` to exit
                if cv2.waitKey(1) & 0xFF == ord('q'):
                    break
            else:
                break
        cap.release()
        del cap
    makePostComparisonClip(vidName, kickFrames, kicks)
    cap = cv2.VideoCapture(f'{OUT_DIR}/{vidName}/Clips/PostDetectionClip.mp4')
    if not cap.isOpened():
        print("Not opening")
    while cap.isOpened():
        ret, frame = cap.read()
        if ret:
            #I can add annotations here
            out.write(frame)
            # Press `q` to exit
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        else:
            break
    cap.release()
    del cap
    out.release()

def makeOutputVid5(vidName):
    cap = cv2.VideoCapture(f'{OUT_DIR}/{vidName}/FinalVideo3.mp4')
    frame_width = int(cap.get(3))
    frame_height = int(cap.get(4))
    video_fps = int(cap.get(5))
    out = cv2.VideoWriter(
        f'{OUT_DIR}/{vidName}/FinalVideo5.mp4', 
        cv2.VideoWriter_fourcc(*'avc1'), 
        video_fps, 
        (frame_width, frame_height), 
    )
    if not cap.isOpened():
            print("Not opening")
    while cap.isOpened():
        ret, frame = cap.read()
        if ret:
            #I can add annotations here
            for i in range(10):
                out.write(frame)
            # Press `q` to exit
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
        else:
            break
    cap.release()
    out.release()


def completeProcess(learnersVid, expertsVid):
    flipRequired1, offset1 = flipNeededAndOffsets(learnersVid)
    MainAngles1 = makeMainVideo(learnersVid, flipRequired1, offset1)
    kickFrames1 = makeClips(learnersVid, MainAngles1, flipRequired1)
    print(kickFrames1)
    flipRequired2, offset2 = flipNeededAndOffsets(expertVid)
    MainAngles2 = makeMainVideo(expertsVid, flipRequired2, offset2)
    kickFrames2 = makeClips(expertsVid, MainAngles2, flipRequired2)
    print(kickFrames2)
    kicks = min(len(kickFrames1), len(kickFrames2))
    master = compareClip(learnersVid, expertsVid, kicks)
    #print(master)
    annotateNormalizedVideos(learnersVid, master, kicks, offset1)
    revertLearnersClips(learnersVid, kicks)
    makeOutputVid1(learnersVid, kickFrames1, kicks)
    makeOutputVid2(learnersVid, kickFrames1, kicks)
    makeOutputVid3(expertsVid, learnersVid, kickFrames2, kicks)
    makeOutputVid4(learnersVid, kickFrames1, kicks)
    makeOutputVid5(learnersVid)
    
    #Will add other output methods


learnerVid = "learnerUnderwater"
expertVid = "expertUnderwater"

completeProcess(learnerVid, expertVid)


Average FPS: 7.627851365493111
STARTING FRAME: 0
CHECKING FRAME 0
136.65726591178125
CHECKING FRAME 1
119.84652569910267
CHECKING FRAME 2
102.55256476888923
CHECKING FRAME 3
101.78909362067509
CHECKING FRAME 4
116.20338109599285
3
CHECKING FRAME 3
101.78909362067509
CHECKING FRAME 4
116.20338109599285
CHECKING FRAME 5
138.44306202129442
CHECKING FRAME 6
156.74640555491644
CHECKING FRAME 7
CHECKING FRAME 8
183.23687458079402
CHECKING FRAME 9
192.3999384853956
CHECKING FRAME 10
190.7325304709059
CHECKING FRAME 11
189.35670798752867
CHECKING FRAME 12
189.63077479217037
CHECKING FRAME 13
CHECKING FRAME 14
186.50883072624927
[14, 9]
CHECKING FRAME 14
186.50883072624927
CHECKING FRAME 15
181.7529579560303
CHECKING FRAME 16
171.41167614518866
CHECKING FRAME 17
159.8221524392439
CHECKING FRAME 18
152.5671576967171
CHECKING FRAME 19
141.49510911459998
CHECKING FRAME 20
125.58972286734988
CHECKING FRAME 21
108.84186475116843
CHECKING FRAME 22
105.49037758175717
CHECKING FRAME 23
114.962050784086

MoviePy - Done !
MoviePy - video ready ../outputs/Trial - 58/learnerUnderwater/FinalVideo3.mp4
